# Diabetes-Vorhersage mit Support Vector Machine (SVM)

Dieses Notebook trainiert ein SVM-Modell, das anhand medizinischer Messwerte vorhersagt, ob eine Person Diabetes hat. Verwendet wird der bekannte [Pima Indians Diabetes Datensatz](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) (768 Patientinnen, 8 Merkmale + Zielvariable `outcome`).

**Vorgehen:**
1. Daten laden & explorieren
2. Fehlende Werte behandeln (im Datensatz als `0` kodiert)
3. Standardisierung der Merkmale
4. Train/Test-Split
5. SVM-Modell trainieren (linearer Kernel)
6. Evaluation (Accuracy, Confusion Matrix, Precision/Recall/F1)
7. Beispielvorhersage für eine einzelne Person


In [38]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [70]:
df_diabetes = pd.read_csv("data/diabetes.csv")

In [71]:
df_diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [72]:
df_diabetes.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [73]:
# Spalten in Kleinbuchstaben
df_diabetes.columns= df_diabetes.columns.str.lower()

In [74]:
# Duplikate
df_diabetes.duplicated().sum()
# Keine Duplikate

np.int64(0)

In [75]:
df_diabetes.isna().sum()
# Keine offensichtlich fehlenden Werte (NaN) -- ABER: siehe nächster Abschnitt!


pregnancies                 0
glucose                     0
bloodpressure               0
skinthickness               0
insulin                     0
bmi                         0
diabetespedigreefunction    0
age                         0
outcome                     0
dtype: int64

### Achtung: Versteckte fehlende Werte

`isna().sum()` zeigt keine fehlenden Werte an. Bei diesem Datensatz sind fehlende Werte bei mehreren Spalten allerdings als **`0`** kodiert -- ein Blutdruck oder Glukosewert von 0 ist medizinisch nicht plausibel. Diese "versteckten" Nullwerte werden hier zunächst identifiziert und dann durch den Median (getrennt nach `outcome`) ersetzt, damit das Modell nicht mit unrealistischen Werten trainiert.

In [76]:
# Spalten, bei denen ein Wert von 0 medizinisch unplausibel ist
zero_invalid_cols = ["glucose", "bloodpressure", "skinthickness", "insulin", "bmi"]

# Anzahl der (versteckten) fehlenden Werte pro Spalte
(df_diabetes[zero_invalid_cols] == 0).sum()


glucose            5
bloodpressure     35
skinthickness    227
insulin          374
bmi               11
dtype: int64

In [77]:
# 0-Werte in diesen Spalten als "echt fehlend" (NaN) markieren
df_diabetes[zero_invalid_cols] = df_diabetes[zero_invalid_cols].replace(0, np.nan)

# Prozentualer Anteil fehlender Werte pro Spalte
(df_diabetes[zero_invalid_cols].isna().mean() * 100).round(1)


glucose           0.7
bloodpressure     4.6
skinthickness    29.6
insulin          48.7
bmi               1.4
dtype: float64

In [78]:
# Imputation: fehlende Werte durch den Median der jeweiligen Outcome-Gruppe ersetzen
# (z.B. wird ein fehlender Insulinwert bei einer diabetischen Person durch den
# Median der diabetischen Personen ersetzt, nicht durch den globalen Median)
for col in zero_invalid_cols:
    df_diabetes[col] = df_diabetes.groupby("outcome")[col]\
        .transform(lambda group: group.fillna(group.median()))

# Kontrolle: keine fehlenden Werte mehr vorhanden
df_diabetes[zero_invalid_cols].isna().sum()


glucose          0
bloodpressure    0
skinthickness    0
insulin          0
bmi              0
dtype: int64

In [79]:
#Datentypen passen überein
df_diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   pregnancies               768 non-null    int64  
 1   glucose                   768 non-null    float64
 2   bloodpressure             768 non-null    float64
 3   skinthickness             768 non-null    float64
 4   insulin                   768 non-null    float64
 5   bmi                       768 non-null    float64
 6   diabetespedigreefunction  768 non-null    float64
 7   age                       768 non-null    int64  
 8   outcome                   768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


In [80]:
# Diabetic cases 0-Non diabetic, 1-Diabetic
df_diabetes["outcome"].value_counts()

outcome
0    500
1    268
Name: count, dtype: int64

In [81]:
# Durchschnittliche Werte 
df_diabetes.groupby("outcome").mean()

,pregnancies,glucose,bloodpressure,skinthickness,insulin,bmi,diabetespedigreefunction,age
outcome,,,,,,,,
0,3.298000,110.622000,70.844000,27.170000,117.172000,30.846000,0.429734,31.190000
1,4.865672,142.302239,75.272388,32.671642,187.615672,35.398507,0.550500,37.067164


In [82]:
# Trennung des Datensatzes vom Label
features=df_diabetes.drop(columns=["outcome"])
target= df_diabetes["outcome"]

In [83]:
# Überblick features
features.head(4)

,pregnancies,glucose,bloodpressure,skinthickness,insulin,bmi,diabetespedigreefunction,age
0,6,148.0,72.0,35.0,169.5,33.6,0.627,50
1,1,85.0,66.0,29.0,102.5,26.6,0.351,31
2,8,183.0,64.0,32.0,169.5,23.3,0.672,32
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21


In [84]:
# Überblick Target
target.head(4)

0    1
1    0
2    1
3    0
Name: outcome, dtype: int64

Daten Standardisieren, da die Range der Spalten sehr stark variert

In [85]:
scaler= StandardScaler()

In [86]:
scaler.fit(features)

StandardScaler()

In [87]:
standardized_data= scaler.transform(features)

In [88]:
# Anschauen ob die Range passt
standardized_data

array([[ 0.63994726,  0.86462486, -0.03218035, ...,  0.16948251,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.20472661, -0.52812374, ..., -0.84854874,
        -0.36506078, -0.19067191],
       [ 1.23388019,  2.01426457, -0.69343821, ..., -1.32847775,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 , -0.02224005, -0.03218035, ..., -0.90672195,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.14199419, -1.02406713, ..., -0.33953311,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.94195182, -0.19749482, ..., -0.2959032 ,
        -0.47378505, -0.87137393]], shape=(768, 8))

In [89]:
# WICHTIG: features (DataFrame mit Spaltennamen) bleibt erhalten,
# damit wir spaeter (z.B. bei der Einzelvorhersage) noch die Spaltennamen kennen.
features_scaled = standardized_data


In [90]:
features_train, features_test, target_train, target_test = train_test_split(
    features_scaled, target, test_size=0.2, stratify=target, random_state=2
)


Model trainieren

In [91]:
classifier = svm.SVC(kernel="linear")

In [92]:
# Trainieren des svm classifier
classifier.fit(features_train, target_train)

SVC(kernel='linear')

Model Evaluation

In [93]:
# Accuracy Score
feature_train_prediction= classifier.predict(features_train)
training_data_accuracy= accuracy_score(feature_train_prediction, target_train)

In [94]:
print("Accuracy score of the training data: ", training_data_accuracy)

Accuracy score of the training data:  0.7850162866449512


Nun wird das Modell auf den test Datensatz angewendet

In [95]:
features_test_prediction=classifier.predict(features_test)
test_data_accuracy=accuracy_score(features_test_prediction, target_test)

In [96]:
print("Accuracy score of the test data: ", test_data_accuracy)

Accuracy score of the test data:  0.7532467532467533


### Genauere Evaluation

Die Klassen sind unbalanciert (500 gesunde vs. 268 diabetische Faelle), daher reicht die Accuracy allein nicht aus, um die Modellgüte zu beurteilen. Confusion Matrix und Classification Report zeigen zusätzlich, wie gut das Modell speziell die diabetischen Fälle erkennt (Recall) und wie viele falsch-positive/negative Vorhersagen es macht.

In [97]:
# Confusion Matrix (Zeilen = tatsaechliche Klasse, Spalten = vorhergesagte Klasse)
cm = confusion_matrix(target_test, features_test_prediction)
pd.DataFrame(
    cm,
    index=["Tatsächlich: Non-diabetic", "Tatsächlich: Diabetic"],
    columns=["Vorhergesagt: Non-diabetic", "Vorhergesagt: Diabetic"]
)


,Vorhergesagt: Non-diabetic,Vorhergesagt: Diabetic
Tatsächlich: Non-diabetic,87,13
Tatsächlich: Diabetic,25,29


In [98]:
# Precision, Recall, F1-Score pro Klasse
print(classification_report(
    target_test,
    features_test_prediction,
    target_names=["Non-diabetic (0)", "Diabetic (1)"]
))


                  precision    recall  f1-score   support

Non-diabetic (0)       0.78      0.87      0.82       100
    Diabetic (1)       0.69      0.54      0.60        54

        accuracy                           0.75       154
       macro avg       0.73      0.70      0.71       154
    weighted avg       0.75      0.75      0.74       154



Vorhersage mit bestimmten Eingabewerten

In [99]:
input_data = (3,158,76,36,245,31.6,0.851,28)

# Als DataFrame mit denselben Spaltennamen wie beim Training uebergeben,
# damit der StandardScaler keine Warnung wegen fehlender Feature-Namen wirft
input_df = pd.DataFrame([input_data], columns=features.columns)

std_data = scaler.transform(input_df)
print(std_data)
prediction = classifier.predict(std_data)

if prediction[0] == 0:
    print("Non diabetic person")
else:
    print("Diabetic person")


[[-0.25095213  1.19309335  0.29844857  0.77773025  1.15951061 -0.12138356
   1.14499856 -0.44593516]]
Diabetic person


## Fazit

Das lineare SVM-Modell erreicht auf den Testdaten eine Accuracy im Bereich von ~75-80 %. Da die Klassen unbalanciert sind, ist besonders der Recall für die Klasse "Diabetic" relevant -- also wie viele der tatsächlich diabetischen Fälle korrekt erkannt werden.

**Mögliche nächste Schritte:**
- Hyperparameter-Tuning (`GridSearchCV`) für `C` und den Kernel (z.B. `rbf` statt `linear`)
- Cross-Validation statt eines einzelnen Train/Test-Splits für eine robustere Schätzung
- Vergleich mit anderen Modellen (z.B. Random Forest, Logistic Regression)
- Feature-Engineering / genauere Analyse, welche Merkmale am stärksten mit `outcome` korrelieren